# 06. Market Basket Analysis & Association Rules

Khai phá tập phổ biến và luật kết hợp bằng thuật toán Apriori/FP-Growth để gợi ý sản phẩm bán kèm (Cross-Selling).


In [1]:
# Setup environment and imports
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src import config


## 1. Chuẩn Bị Ma Trận Giỏ Hàng (One-Hot Encoded)


In [2]:
from src.association_rules import prepare_basket_matrix
from src.data_loader import load_interim_data
cleaned_df = load_interim_data('cleaned_transactions.csv')
basket = prepare_basket_matrix(cleaned_df)
basket.head()

Đang chuẩn bị ma trận giỏ hàng (quá trình này có thể mất chút thời gian)...
[OK] Đã tạo xong ma trận giỏ hàng với kích thước: (19773, 4018)


Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 2. Khai Phá Tập Phổ Biến & Sinh Luật Kết Hợp


In [3]:
from src.association_rules import extract_association_rules
rules = extract_association_rules(basket, min_support=0.02, min_threshold=0.5)
rules.head(10)

Đang chạy FP-Growth với min_support=0.02...
Tìm thấy 381 tập phổ biến. Đang sinh luật...
[OK] Đã sinh thành công 60 luật kết hợp.


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED }),0.049563,0.053153,0.032367,0.653061,12.286374,1.0,0.029733,2.729146,0.966512,0.460101,0.633585,0.631003
1,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.053153,0.049563,0.032367,0.608944,12.286374,1.0,0.029733,2.430437,0.970177,0.460101,0.588551,0.631003
2,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({ALARM CLOCK BAKELIKE RED }),0.039599,0.053153,0.023770,0.600255,11.292912,1.0,0.021665,2.368629,0.949030,0.344575,0.577815,0.523724
3,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.039599,0.049563,0.021140,0.533844,10.771124,1.0,0.019177,2.038884,0.944563,0.310781,0.509536,0.480187
4,frozenset({WOODEN FRAME ANTIQUE WHITE }),frozenset({WOODEN PICTURE FRAME WHITE FINISH}),0.049107,0.055631,0.027259,0.555098,9.978136,1.0,0.024527,2.122643,0.946249,0.351828,0.528889,0.522549
5,frozenset({RED HANGING HEART T-LIGHT HOLDER}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.037475,0.114095,0.025034,0.668016,5.854913,1.0,0.020758,2.668519,0.861488,0.197842,0.625260,0.443716
6,frozenset({JUMBO BAG PINK POLKADOT}),frozenset({JUMBO BAG RED RETROSPOT}),0.061599,0.105649,0.041724,0.677340,6.411222,1.0,0.035216,2.771805,0.899427,0.332393,0.639224,0.536133
7,frozenset({JUMBO STORAGE BAG SUKI}),frozenset({JUMBO BAG RED RETROSPOT}),0.059880,0.105649,0.036616,0.611486,5.787900,1.0,0.030289,2.301981,0.879915,0.284033,0.565592,0.479032
8,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO STOR...",frozenset({JUMBO BAG PINK POLKADOT}),0.036616,0.061599,0.020887,0.570442,9.260550,1.0,0.018632,2.184573,0.925918,0.270111,0.542245,0.454761
9,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO BAG ...",frozenset({JUMBO STORAGE BAG SUKI}),0.041724,0.059880,0.020887,0.500606,8.360206,1.0,0.018389,1.882523,0.918718,0.258772,0.468798,0.424712


## 3. Lọc & Sắp Xếp Luật Theo Lift


In [4]:
top_rules = rules.sort_values(by='lift', ascending=False)
top_rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
50,"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.038790,0.038689,0.027361,0.705346,18.231107,1.0,0.025860,3.262502,0.983291,0.545913,0.693487,0.706268
51,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",0.038689,0.038790,0.027361,0.707190,18.231107,1.0,0.025860,3.282703,0.983187,0.545913,0.695373,0.706268
49,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.030243,0.051231,0.027361,0.904682,17.658719,1.0,0.025811,9.953747,0.972791,0.505607,0.899535,0.719370
52,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",0.051231,0.030243,0.027361,0.534057,17.658719,1.0,0.025811,2.081279,0.994311,0.505607,0.519526,0.719370
45,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.038689,0.051231,0.031963,0.826144,16.125707,1.0,0.029981,5.457202,0.975738,0.551483,0.816756,0.725017
44,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({PINK REGENCY TEACUP AND SAUCER}),0.051231,0.038689,0.031963,0.623889,16.125707,1.0,0.029981,2.555926,0.988637,0.551483,0.608752,0.725017
53,frozenset({ROSES REGENCY TEACUP AND SAUCER }),"frozenset({GREEN REGENCY TEACUP AND SAUCER, PI...",0.053861,0.031963,0.027361,0.507981,15.892900,1.0,0.025639,1.967480,0.990424,0.467993,0.491736,0.681997
48,"frozenset({GREEN REGENCY TEACUP AND SAUCER, PI...",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.031963,0.053861,0.027361,0.856013,15.892900,1.0,0.025639,6.570985,0.968019,0.467993,0.847816,0.681997
58,frozenset({GARDENERS KNEELING PAD KEEP CALM }),frozenset({GARDENERS KNEELING PAD CUP OF TEA }),0.046174,0.038335,0.027613,0.598028,15.600023,1.0,0.025843,2.392371,0.981204,0.485333,0.582005,0.659173
59,frozenset({GARDENERS KNEELING PAD CUP OF TEA }),frozenset({GARDENERS KNEELING PAD KEEP CALM }),0.038335,0.046174,0.027613,0.720317,15.600023,1.0,0.025843,3.410378,0.973205,0.485333,0.706777,0.659173


## 4. Lưu Danh Sách Luật Kết Hợp


In [5]:
from src.data_loader import save_rules
save_rules(top_rules, 'association_rules.csv')

[OK] Đã lưu thành công danh sách luật kết hợp tại: D:\HK1 2026-2027\KTDL\Data-mining\outputs\rules\association_rules.csv
